# Janus-1.3B Fine-Tuning (Colab) — QLoRA/LoRA

> This notebook is designed to run on **Google Colab GPU** (T4/A10/A100). It fine-tunes a *small* Janus-style 1–1.3B LLM using **QLoRA (4-bit)** so you can train cheaply and export a lightweight adapter.

## What you get
- A reproducible training pipeline (dataset → preprocess → train → save).
- Outputs saved to Google Drive.
- Optional: merge LoRA into base model and push to Hugging Face.

## Before you start
1. Open this notebook in Google Colab.
2. Set runtime to **GPU**: `Runtime → Change runtime type → GPU`.
3. Make sure you have a Hugging Face account + token if you want to download gated models or push results.

## Reality check (why QLoRA)
- Colab GPU is the practical way to fine-tune; Intel HD 520 laptops are fine for **testing inference**, not training.
- QLoRA keeps VRAM low by loading the base model in **4-bit** and training only small LoRA weights.
- If you get out-of-memory (OOM), reduce `MAX_SEQ_LEN`, `BATCH_SIZE`, or `GRAD_ACCUM_STEPS`.

In [ ]:
#@title 1) Check GPU / environment
import os, sys, platform, textwrap
import torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('No GPU detected. In Colab: Runtime → Change runtime type → GPU')

In [ ]:
#@title 2) Install dependencies (Colab)
# If you re-run this cell, use Runtime → Restart runtime afterwards.
# Janus / Janus-Pro typically needs timm + attrdict for the vision encoder stack.
!pip -q install -U "transformers>=4.41" "accelerate>=0.33" "datasets>=2.20" "peft>=0.12" "trl>=0.9" "bitsandbytes>=0.43" "sentencepiece" "einops" "pillow" "torchvision" "huggingface_hub" "timm>=0.9.16" "attrdict"

# Optional speedups
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print('Done installing.')

In [ ]:
#@title 3) Imports
from dataclasses import dataclass
from typing import Dict, List, Optional, Any
import json
import numpy as np
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
 )
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from PIL import Image

## 4) Mount Google Drive (recommended)
This saves your checkpoints even if the Colab runtime disconnects.

In [ ]:
#@title 4) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/BrainInk-Janus-Finetune'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print('Project dir:', PROJECT_DIR)

## 5) Configure your run
Fill these in once, then you can re-run training quickly.

In [ ]:
#@title 5) Run configuration
@dataclass
class RunConfig:
    # --- Mode ---
    # 'text' = text-only instruction tuning (original path)
    # 'vlm'  = multimodal grading (image + rubric + output)
    MODE: str = "vlm"

    # --- Model ---
    # Put the Hugging Face model id here (examples: 'org/model')
    # For Janus multimodal models, you may need TRUST_REMOTE_CODE=True.
    BASE_MODEL: str = "deepseek-ai/Janus-Pro-1B"
    TRUST_REMOTE_CODE: bool = True

    # --- Data ---
    # Option A: Hugging Face dataset name (leave empty to use local files)
    HF_DATASET: str = ""
    HF_SPLIT: str = "train"
    # Option B: Local jsonl path in Colab (you can upload it or keep it in Drive)
    LOCAL_TRAIN_JSONL: str = ""  # e.g. f"{PROJECT_DIR}/data/train.jsonl"
    LOCAL_EVAL_JSONL: str = ""   # optional

    # Data format (text-only path): choose ONE
    # - 'alpaca': expects keys: instruction, input(optional), output
    # - 'chatml': expects key: messages (list of {role, content})
    # - 'text': expects key: text (already formatted prompt+response)
    DATA_FORMAT: str = "alpaca"

    # Multimodal grading columns (vlm path)
    # Required for vlm: IMAGE_PATH_COL + RUBRIC_COL + RESPONSE_COL
    IMAGE_PATH_COL: str = "image_path"
    # If you store multiple images per sample, set IMAGES_COL to e.g. 'image_paths' (list[str]) and leave IMAGE_PATH_COL unused
    IMAGES_COL: str = ""
    PROMPT_COL: str = "prompt"            # e.g. question / assignment instructions (optional but recommended)
    RUBRIC_COL: str = "rubric"            # rubric text provided by user/teacher
    CONTEXT_COL: str = "context"          # optional (class, grade level, topic, etc.)
    RESPONSE_COL: str = "grade_json"       # the target output (string)
    IMAGE_ROOT: str = ""                  # optional prefix if paths are relative

    # --- Training ---
    SEED: int = 42
    MAX_SEQ_LEN: int = 1024
    BATCH_SIZE: int = 1
    GRAD_ACCUM_STEPS: int = 8
    EPOCHS: float = 1.0
    LEARNING_RATE: float = 2e-4
    WARMUP_RATIO: float = 0.03
    WEIGHT_DECAY: float = 0.0
    LOGGING_STEPS: int = 10
    SAVE_STEPS: int = 100

    # --- QLoRA / LoRA ---
    LOAD_IN_4BIT: bool = True
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05
    LORA_TARGET_MODULES: Optional[List[str]] = None

    # --- Output ---
    RUN_NAME: str = "janus_run"
    OUTPUT_DIR: str = f"{PROJECT_DIR}/outputs"

cfg = RunConfig()
print(cfg)

## 6) Hugging Face login (optional, but recommended)
Needed if your base model is gated/private, or if you plan to push outputs to the Hub.

In [ ]:
#@title 6) Hugging Face login (optional)
import os

# If you want, you can set it in Colab Secrets instead of pasting here.
# Left sidebar → Secrets → add HF_TOKEN
HF_TOKEN = os.environ.get('HF_TOKEN', '')

try:
    from huggingface_hub import login
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print('Logged in using HF_TOKEN from environment.')
    else:
        print('HF_TOKEN not found. If your model is public, you can continue without login.')
        print('If gated/private: set HF_TOKEN in Colab Secrets and re-run this cell.')
except Exception as e:
    print('Hugging Face login skipped:', e)

## 7) Dataset: format + loader
This notebook supports three dataset formats. Pick one by setting `cfg.DATA_FORMAT`:

- `alpaca`: JSON/JSONL with `instruction`, optional `input`, and `output`
- `chatml`: JSON/JSONL with `messages` = list of `{role: 'system'|'user'|'assistant', content: '...'}`
- `text`: JSON/JSONL with `text` already containing the final prompt+answer (recommended for maximum control)

You can load either:
- A Hugging Face dataset (`cfg.HF_DATASET`)
- A local JSONL file in Drive (`cfg.LOCAL_TRAIN_JSONL`)

In [ ]:
#@title 7A) (Optional) Create a tiny sample dataset in Drive
import json, os
sample_path = f"{PROJECT_DIR}/data/sample_train.jsonl"
os.makedirs(os.path.dirname(sample_path), exist_ok=True)

samples = [
    {"instruction": "Write a short welcome message for BrainInk.", "input": "", "output": "Welcome to BrainInk — where ideas become ink. Let's build something great."},
    {"instruction": "Rewrite the text in a friendlier tone.", "input": "We will proceed with the plan.", "output": "Sounds good — we’ll move forward with the plan."},
    {"instruction": "Answer as a helpful tutor.", "input": "What is gradient accumulation?", "output": "Gradient accumulation lets you simulate a larger batch size by summing gradients over multiple forward/backward passes before taking an optimizer step."},
 ]

with open(sample_path, "w", encoding="utf-8") as f:
    for row in samples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print('Wrote sample dataset:', sample_path)
print('Tip: set cfg.LOCAL_TRAIN_JSONL to this path to test training quickly.')

In [ ]:
#@title 7B) Prompt formatting + dataset loader
import re
from datasets import Dataset, DatasetDict

def format_alpaca(example: Dict, eos: str) -> str:
    instruction = (example.get('instruction') or '').strip()
    input_text = (example.get('input') or '').strip()
    output_text = (example.get('output') or '').strip()
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output_text}{eos}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output_text}{eos}"
    return prompt

def format_chatml(example: Dict, eos: str) -> str:
    msgs = example.get('messages')
    if not isinstance(msgs, list):
        raise ValueError("chatml format expects 'messages' list")
    chunks = []
    for m in msgs:
        role = (m.get('role') or '').strip()
        content = (m.get('content') or '').strip()
        if not role or content is None:
            continue
        # Simple ChatML-like rendering
        chunks.append(f"<|{role}|>\n{content}\n")
    chunks.append(f"<|assistant|>\n{eos}")
    return ''.join(chunks)

def format_text(example: Dict, eos: str) -> str:
    text = example.get('text')
    if text is None:
        raise ValueError("text format expects 'text'")
    text = str(text)
    if not text.endswith(eos):
        text = text + eos
    return text

def load_any_dataset(cfg: RunConfig):
    if cfg.HF_DATASET.strip():
        ds = load_dataset(cfg.HF_DATASET, split=cfg.HF_SPLIT)
        return DatasetDict(train=ds)

    if cfg.LOCAL_TRAIN_JSONL.strip():
        data_files = {'train': cfg.LOCAL_TRAIN_JSONL}
        if cfg.LOCAL_EVAL_JSONL.strip():
            data_files['test'] = cfg.LOCAL_EVAL_JSONL
        ds = load_dataset('json', data_files=data_files)
        return ds

    raise ValueError('Set either cfg.HF_DATASET or cfg.LOCAL_TRAIN_JSONL')

def build_text_dataset(raw_ds: DatasetDict, tokenizer, cfg: RunConfig) -> DatasetDict:
    eos = tokenizer.eos_token or ''
    if cfg.DATA_FORMAT == 'alpaca':
        fn = lambda ex: {'text': format_alpaca(ex, eos)}
    elif cfg.DATA_FORMAT == 'chatml':
        fn = lambda ex: {'text': format_chatml(ex, eos)}
    elif cfg.DATA_FORMAT == 'text':
        fn = lambda ex: {'text': format_text(ex, eos)}
    else:
        raise ValueError(f'Unknown cfg.DATA_FORMAT: {cfg.DATA_FORMAT}')

    cols = list(raw_ds['train'].features.keys())
    text_ds = raw_ds.map(fn, remove_columns=cols)
    return text_ds

print('Dataset utilities ready.')

## 8) Load tokenizer + base model in 4-bit (QLoRA)
This is the core step that makes fine-tuning possible on Colab GPUs.

In [ ]:
#@title 8) Load tokenizer + model (4-bit)
set_seed(cfg.SEED)

tokenizer = AutoTokenizer.from_pretrained(cfg.BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = None
if cfg.LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    )

model = AutoModelForCausalLM.from_pretrained(
    cfg.BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)
model.config.use_cache = False

# Prepare for k-bit training (enables gradient checkpointing-friendly settings)
model = prepare_model_for_kbit_training(model)

def guess_lora_targets(model) -> List[str]:
    # Common for Llama-like architectures
    preferred = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    names = set()
    for n, _ in model.named_modules():
        leaf = n.split('.')[-1]
        if leaf in preferred:
            names.add(leaf)
    if names:
        return sorted(names)
    # Fallback: pick any linear-ish projection modules by name heuristic
    for n, _ in model.named_modules():
        leaf = n.split('.')[-1]
        if any(k in leaf for k in ['proj','fc','wq','wk','wv','wo']):
            names.add(leaf)
    return sorted(names)

target_modules = cfg.LORA_TARGET_MODULES or guess_lora_targets(model)
print('LoRA target modules:', target_modules)

peft_config = LoraConfig(
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=target_modules,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 9) Build the training dataset (`text` field)
This converts your dataset into a single `text` column containing prompt+response pairs.

In [ ]:
#@title 9) Load + format dataset
raw_ds = load_any_dataset(cfg)
print(raw_ds)

ds = build_text_dataset(raw_ds, tokenizer, cfg)
print(ds)

print('Sample formatted text:\n')
print(ds['train'][0]['text'][:800])

## 10) Train (SFT with QLoRA)
If you hit OOM: lower `cfg.MAX_SEQ_LEN`, increase `cfg.GRAD_ACCUM_STEPS`, or keep `BATCH_SIZE=1`.

In [ ]:
#@title 10) Train
import math, os

run_output_dir = os.path.join(cfg.OUTPUT_DIR, cfg.RUN_NAME)
os.makedirs(run_output_dir, exist_ok=True)
print('Run output:', run_output_dir)

training_args = TrainingArguments(
    output_dir=run_output_dir,
    per_device_train_batch_size=cfg.BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRAD_ACCUM_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    num_train_epochs=cfg.EPOCHS,
    warmup_ratio=cfg.WARMUP_RATIO,
    weight_decay=cfg.WEIGHT_DECAY,
    logging_steps=cfg.LOGGING_STEPS,
    save_steps=cfg.SAVE_STEPS,
    save_total_limit=2,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    fp16=(torch.cuda.is_available() and not torch.cuda.is_bf16_supported()),
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    report_to='none',
    run_name=cfg.RUN_NAME,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
 )

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds.get('test', None),
    dataset_text_field='text',
    max_seq_length=cfg.MAX_SEQ_LEN,
    args=training_args,
 )

train_result = trainer.train()
print(train_result)

## 11) Save outputs (LoRA adapter)
This saves a small adapter you can load later with the base model. It’s usually only tens/hundreds of MB.

In [ ]:
#@title 11) Save adapter + tokenizer
adapter_dir = os.path.join(run_output_dir, 'adapter')
os.makedirs(adapter_dir, exist_ok=True)

trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print('Saved adapter to:', adapter_dir)
print('Files:', os.listdir(adapter_dir)[:10])

## 12) Inference smoke test (base + adapter)
This reloads the adapter you just saved and generates a quick response.

In [ ]:
#@title 12) Inference smoke test
import torch
from peft import PeftModel

prompt = "### Instruction:\nWrite a 2-sentence pitch for BrainInk.\n\n### Response:\n"

# Reload base + adapter for a clean test (uses same quantization settings as earlier)
test_tokenizer = AutoTokenizer.from_pretrained(adapter_dir, use_fast=True)
if test_tokenizer.pad_token is None:
    test_tokenizer.pad_token = test_tokenizer.eos_token
test_tokenizer.padding_side = 'left'

test_bnb = bnb_config if cfg.LOAD_IN_4BIT else None
base = AutoModelForCausalLM.from_pretrained(
    cfg.BASE_MODEL,
    quantization_config=test_bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
 )
base.eval()

adapted = PeftModel.from_pretrained(base, adapter_dir)
adapted.eval()

inputs = test_tokenizer(prompt, return_tensors='pt').to(adapted.device)
with torch.no_grad():
    out = adapted.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=test_tokenizer.eos_token_id,
    )

print(test_tokenizer.decode(out[0], skip_special_tokens=True))

## 13) (Optional) Merge LoRA into the base model
You only need this if you want a single “merged” model folder. This usually requires loading the base model in FP16/BF16 (not 4-bit) and may need more VRAM.

If you just want to run inference later, keeping **base + adapter** is totally fine and often preferred.

In [ ]:
#@title 13) Merge adapter into base (optional)
import os, torch
from peft import PeftModel

# Set True if you want to run the merge now
DO_MERGE = False

merged_dir = os.path.join(run_output_dir, 'merged_model')
os.makedirs(merged_dir, exist_ok=True)

if not DO_MERGE:
    print('DO_MERGE=False → skipping merge. Set DO_MERGE=True to run.')
else:
    # Reload base in higher precision (no 4-bit) to allow merging weights cleanly
    merge_base = AutoModelForCausalLM.from_pretrained(
        cfg.BASE_MODEL,
        device_map='auto',
        torch_dtype=torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16,
    )
    merge_base.eval()

    merged = PeftModel.from_pretrained(merge_base, adapter_dir)
    merged = merged.merge_and_unload()
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    print('Merged model saved to:', merged_dir)

## 14) (Optional) Push to Hugging Face Hub
You can push either:
- The adapter folder (recommended): small upload, keeps licensing clean (base model stays separate)
- The merged model folder (bigger upload)

In [ ]:
#@title 14) Push adapter/merged model to Hugging Face (optional)
import os
from huggingface_hub import HfApi, create_repo

# Fill these in if you want to push
HF_USERNAME_OR_ORG = ""   # e.g. 'yourname' or 'yourorg'
HF_REPO_NAME = "brainink-janus-qlora"
PUSH_WHAT = "adapter"     # 'adapter' or 'merged'
PRIVATE_REPO = True

if not HF_USERNAME_OR_ORG:
    print('Set HF_USERNAME_OR_ORG to push.')
else:
    repo_id = f"{HF_USERNAME_OR_ORG}/{HF_REPO_NAME}"
    create_repo(repo_id, private=PRIVATE_REPO, exist_ok=True)
    api = HfApi()

    folder = adapter_dir if PUSH_WHAT == 'adapter' else merged_dir
    if not os.path.exists(folder):
        raise FileNotFoundError(f'Folder not found: {folder}')

    print('Pushing folder:', folder)
    print('Repo:', repo_id)
    api.upload_folder(
        repo_id=repo_id,
        folder_path=folder,
        path_in_repo=".",
        commit_message=f"Upload {PUSH_WHAT} from {cfg.RUN_NAME}",
    )
    print('Done.')

## 15) Troubleshooting (most common fixes)
- **OOM / CUDA out of memory**: set `cfg.MAX_SEQ_LEN=512`, keep `cfg.BATCH_SIZE=1`, increase `cfg.GRAD_ACCUM_STEPS` (8→16).
- **Model won’t download**: login with `HF_TOKEN` in Colab Secrets, and make sure you accepted the model license on Hugging Face.
- **Slow training**: reduce sequence length; T4 GPUs are slower than A100.
- **Bad outputs**: you likely need more/better data, or more epochs. Start with small runs to validate formatting.

# Multimodal (Janus-Pro) fine-tuning path (image + rubric grading)
This section adds a **true multimodal** training loop for grading handwritten work from images (plots, steps, diagrams) using a rubric.

This notebook is tailored to `deepseek-ai/Janus-Pro-1B` and follows the **official Janus-Pro pipeline**:
- Load the model via `AutoModelForCausalLM(..., trust_remote_code=True)`
- Use `VLChatProcessor` to build multimodal inputs
- Call `prepare_inputs_embeds(...)`, then train / generate with `language_model`
- Train only a small **LoRA adapter** on the language model (`vl_gpt.language_model`)

> If you switch to a different Janus checkpoint, its API may differ — update the loader accordingly.

## VLM Dataset schema (JSONL)
Each JSONL row should look like this (recommended minimum):
- `image_path`: path to the student submission image (handwritten work)
- `prompt`: the assignment instructions / question
- `rubric`: the rubric text (criteria + points)
- `grade_json`: the *target output* as a JSON string (model learns to output this)

Example row:
```json
{
  "image_path": "data/submissions/s1.png",
  "prompt": "Plot y = 2x + 1 for x in [-2,2]. Label intercept.",
  "rubric": "Total 10 points. (1) Correct slope (0-3). (2) Correct intercept (0-3). (3) Line drawn correctly (0-3). (4) Axes labeled (0-1).",
  "grade_json": "{\"overall\":8,\"criteria\":[{\"name\":\"slope\",\"score\":3,\"max\":3,\"evidence\":\"line rises 2 per 1\"},...],\"feedback\":\"Label the y-intercept more clearly.\"}"
}
```

Notes:
- Keep `grade_json` deterministic and strict JSON so it’s easy to parse in your app.
- If you have multiple images per submission, store `image_paths` as a list and set `cfg.IMAGES_COL='image_paths'`.

## VLM data alignment (how images + rubric + grades “match”)
For multimodal fine-tuning you need **(A)** the image files and **(B)** a JSONL file where each line points to the image and includes the rubric + the “gold” grading output.

Think of it like this: **one JSONL line = one training example**
- `image_path` (or `image_paths`) locates the student work image(s) on disk
- `prompt` describes the assignment / question
- `rubric` is the grading criteria and point scheme
- `grade_json` is the correct final grade output you want the model to learn (STRICT JSON string)

> During training, the loader reads the JSONL row, opens the image at that path, and trains the model to output `grade_json` given the image + rubric + prompt.

In [1]:
#@title VLM 0) (Optional) Create a starter VLM dataset + folder layout in Drive
import os, json
from PIL import Image, ImageDraw

# Recommended layout under your PROJECT_DIR
DATA_DIR = f"{PROJECT_DIR}/data"
IMG_DIR = f"{DATA_DIR}/images"
os.makedirs(IMG_DIR, exist_ok=True)

# Create a tiny placeholder image so the pipeline can run end-to-end
img_path = os.path.join(IMG_DIR, "sample_001.png")
if not os.path.exists(img_path):
    im = Image.new("RGB", (768, 768), "white")
    d = ImageDraw.Draw(im)
    d.text((30, 30), "BrainInk sample\n(handwritten mock)", fill="black")
    d.line((60, 200, 700, 200), fill="black", width=4)
    d.text((60, 240), "y = 2x + 1", fill="black")
    im.save(img_path)

# One JSONL line = one (image, rubric, target) training example
train_jsonl = os.path.join(DATA_DIR, "vlm_train.jsonl")
example = {
    "image_path": img_path,
    "prompt": "Plot y = 2x + 1 for x in [-2,2]. Label intercept.",
    "rubric": "Total 10 points. (1) Correct slope (0-3). (2) Correct intercept (0-3). (3) Line drawn correctly (0-3). (4) Axes labeled (0-1).",
    # IMPORTANT: grade_json is a STRING containing JSON (so your app can parse it reliably)
    "grade_json": json.dumps({
        "overall": 8,
        "criteria": [
            {"name": "slope", "score": 3, "max": 3, "evidence": "Line rises ~2 for every +1 in x"},
            {"name": "intercept", "score": 2, "max": 3, "evidence": "Intercept appears near 1 but not clearly labeled"},
            {"name": "line", "score": 3, "max": 3, "evidence": "Single straight line consistent with y=2x+1"},
            {"name": "axes", "score": 0, "max": 1, "evidence": "Axes labels missing"},
        ],
        "feedback": "Good slope and line. Clearly label the y-intercept and axes next time.",
    }, ensure_ascii=False),
}

with open(train_jsonl, "w", encoding="utf-8") as f:
    f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Wrote:")
print(" - image:", img_path)
print(" - jsonl:", train_jsonl)
print("\nSet these in your config (Cell 9):")
print(f"cfg.LOCAL_TRAIN_JSONL = '{train_jsonl}'")
print(f"cfg.IMAGE_ROOT = ''  # not needed because image_path is absolute")

NameError: name 'PROJECT_DIR' is not defined

In [ ]:
#@title VLM 1) Load multimodal dataset (images + rubric + target)
from datasets import DatasetDict

def _resolve_path(p: str) -> str:
    p = str(p)
    if cfg.IMAGE_ROOT and not os.path.isabs(p):
        return os.path.join(cfg.IMAGE_ROOT, p)
    return p

def load_vlm_dataset(cfg: RunConfig) -> DatasetDict:
    if cfg.HF_DATASET.strip():
        ds = load_dataset(cfg.HF_DATASET, split=cfg.HF_SPLIT)
        return DatasetDict(train=ds)
    if cfg.LOCAL_TRAIN_JSONL.strip():
        data_files = {'train': cfg.LOCAL_TRAIN_JSONL}
        if cfg.LOCAL_EVAL_JSONL.strip():
            data_files['test'] = cfg.LOCAL_EVAL_JSONL
        return load_dataset('json', data_files=data_files)
    raise ValueError('Set either cfg.HF_DATASET or cfg.LOCAL_TRAIN_JSONL')

def build_vlm_example(ex: Dict[str, Any]) -> Dict[str, Any]:
    # Keep dataset light: store image paths; load PIL in the trainer.
    paths: List[str] = []
    if cfg.IMAGES_COL.strip():
        raw_paths = ex.get(cfg.IMAGES_COL, []) or []
        paths = [_resolve_path(p) for p in raw_paths]
    else:
        p = ex.get(cfg.IMAGE_PATH_COL, '')
        paths = [_resolve_path(p)]

    prompt = (ex.get(cfg.PROMPT_COL, '') or '').strip()
    rubric = (ex.get(cfg.RUBRIC_COL, '') or '').strip()
    context = (ex.get(cfg.CONTEXT_COL, '') or '').strip()
    target = (ex.get(cfg.RESPONSE_COL, '') or '').strip()

    sys = (
        "You are an expert teacher and grader. "
        "Grade strictly using ONLY the provided rubric. "
        "Return STRICT JSON only (no markdown)."
    )

    user = ""
    if context:
        user += f"Context:\n{context}\n\n"
    user += f"Assignment:\n{prompt}\n\nRubric:\n{rubric}\n\n"
    user += "Student work is provided as image(s).\n\n"
    user += "Task: Produce a JSON grade with per-criterion scores, evidence, and feedback."

    # Janus expects the <image_placeholder> token in the user content.
    # We'll put it at the top and then include the rubric prompt.
    prompt_text = f"<image_placeholder>\n{user}"

    return {
        'image_paths': paths,
        'prompt_text': prompt_text,
        'target_text': target,
        'system_text': sys,
    }

raw_vlm = load_vlm_dataset(cfg)
vlm = raw_vlm.map(build_vlm_example)
print(vlm)
print('Example keys:', vlm['train'].column_names)
print('Prompt preview:\n', vlm['train'][0]['prompt_text'][:500])
print('Target preview:\n', vlm['train'][0]['target_text'][:500])

In [ ]:
#@title VLM 2) Load Janus-Pro-1B model + VLChatProcessor (DeepSeek official API)
set_seed(cfg.SEED)

# For Janus-Pro-1B, Transformers uses a custom MultiModalityCausalLM behind AutoModelForCausalLM(trust_remote_code=True).
# We follow the official inference pattern used in DeepSeek's Janus demo code.

MODEL_PATH = cfg.BASE_MODEL  # e.g. 'deepseek-ai/Janus-Pro-1B'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
language_config = getattr(config, 'language_config', None)
if language_config is not None:
    # DeepSeek demos set eager attention for stability
    try:
        language_config._attn_implementation = 'eager'
    except Exception:
        pass

# Try 4-bit if requested; fall back to bf16/fp16 if this checkpoint doesn't support BnB cleanly.
bnb_cfg = None
use_4bit = bool(cfg.LOAD_IN_4BIT) and device == 'cuda'
if use_4bit:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

try:
    vl_gpt = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        language_config=language_config,
        device_map='auto' if device == 'cuda' else None,
        quantization_config=bnb_cfg,
        torch_dtype=dtype if device == 'cuda' else None,
    )
except Exception as e:
    print('4-bit load failed; falling back to non-quantized bf16/fp16. Error:', e)
    vl_gpt = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        language_config=language_config,
        torch_dtype=dtype if device == 'cuda' else None,
    )
    if device == 'cuda':
        vl_gpt = vl_gpt.cuda()

vl_gpt.eval()

# Janus provides VLChatProcessor in its remote code package
from janus.models import VLChatProcessor
vl_chat_processor = VLChatProcessor.from_pretrained(MODEL_PATH)
tokenizer = vl_chat_processor.tokenizer

# Apply LoRA to the language model component (this is what generates the grading JSON)
if use_4bit and hasattr(vl_gpt, 'language_model'):
    vl_gpt.language_model = prepare_model_for_kbit_training(vl_gpt.language_model)

def guess_lora_targets_llm(model) -> List[str]:
    preferred = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    found = set()
    for n, _ in model.named_modules():
        leaf = n.split('.')[-1]
        if leaf in preferred:
            found.add(leaf)
    return sorted(found) if found else ['q_proj','v_proj']

lora_targets = cfg.LORA_TARGET_MODULES or guess_lora_targets_llm(vl_gpt.language_model)
print('LoRA target modules:', lora_targets)

lora_cfg = LoraConfig(
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=lora_targets,
 )

vl_gpt.language_model = get_peft_model(vl_gpt.language_model, lora_cfg)
vl_gpt.language_model.print_trainable_parameters()

In [ ]:
#@title VLM 3) Janus rubric-grading Trainer (multimodal loss)
import torch
from PIL import Image

def _open_first_image(paths: List[str]) -> Image.Image:
    if not paths:
        raise ValueError('No image paths in sample')
    return Image.open(paths[0]).convert('RGB')

def _build_user_block(sample: Dict[str, Any]) -> str:
    # prompt_text already includes <image_placeholder> and the rubric instructions
    return sample['prompt_text']

def _build_system_block(sample: Dict[str, Any]) -> str:
    return sample.get('system_text', "You are an expert teacher and grader. Return STRICT JSON only.")

def _conversation(user_text: str, assistant_text: str, pil_image: Image.Image):
    # DeepSeek Janus demos use roles 'User'/'Assistant'
    return [
        {
            'role': 'User',
            'content': user_text,
            'images': [pil_image],
        },
        {
            'role': 'Assistant',
            'content': assistant_text,
        },
    ]

def _prepare_one(sample: Dict[str, Any], assistant_text: str):
    img = _open_first_image(sample['image_paths'])
    sys = _build_system_block(sample).strip()
    user = _build_user_block(sample).strip()

    # Put the system instruction in-band; VLChatProcessor will apply the SFT template.
    user_text = f"{sys}\n\n{user}"

    conv = _conversation(user_text=user_text, assistant_text=assistant_text, pil_image=img)
    prepared = vl_chat_processor(conversations=conv, images=[img], force_batchify=True)
    prepared = prepared.to(device, dtype=dtype) if hasattr(prepared, 'to') else prepared
    return prepared

class JanusRubricTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # data_collator returns list[dict] of raw samples
        batch = inputs
        if isinstance(inputs, dict):
            batch = inputs.get('batch', [])

        losses = []
        for sample in batch:
            prepared_full = _prepare_one(sample, assistant_text=sample['target_text'])
            prepared_prompt = _prepare_one(sample, assistant_text='')

            inputs_embeds = model.prepare_inputs_embeds(**prepared_full)

            if not hasattr(prepared_full, 'input_ids'):
                raise RuntimeError('VLChatProcessor output missing input_ids; cannot build labels for training.')

            labels = prepared_full.input_ids.clone()
            prompt_len = int(prepared_prompt.attention_mask[0].sum().item())
            labels[:, :prompt_len] = -100

            out = model.language_model(
                inputs_embeds=inputs_embeds,
                attention_mask=prepared_full.attention_mask,
                labels=labels,
                use_cache=False,
            )
            losses.append(out.loss)

        loss = torch.stack(losses).mean()
        return (loss, None) if return_outputs else loss

def identity_collator(features):
    return features

print('JanusRubricTrainer ready. Recommended: cfg.BATCH_SIZE=1 for multimodal training.')

In [ ]:
#@title VLM 4) Train Janus multimodal grader (LoRA/QLoRA)
import os

if cfg.MODE != 'vlm':
    print("cfg.MODE is not 'vlm'. Set cfg.MODE='vlm' in the config cell if you want multimodal training.")
else:
    if cfg.BATCH_SIZE != 1:
        print('Note: for Janus multimodal training, cfg.BATCH_SIZE=1 is strongly recommended.')

    run_output_dir = os.path.join(cfg.OUTPUT_DIR, cfg.RUN_NAME + "_januspro_vlm")
    os.makedirs(run_output_dir, exist_ok=True)
    print('VLM run output:', run_output_dir)

    args = TrainingArguments(
        output_dir=run_output_dir,
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=cfg.GRAD_ACCUM_STEPS,
        learning_rate=cfg.LEARNING_RATE,
        num_train_epochs=cfg.EPOCHS,
        warmup_ratio=cfg.WARMUP_RATIO,
        weight_decay=cfg.WEIGHT_DECAY,
        logging_steps=cfg.LOGGING_STEPS,
        save_steps=cfg.SAVE_STEPS,
        save_total_limit=2,
        lr_scheduler_type='cosine',
        optim='paged_adamw_8bit' if cfg.LOAD_IN_4BIT else 'adamw_torch',
        fp16=(torch.cuda.is_available() and not torch.cuda.is_bf16_supported()),
        bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
        report_to='none',
        run_name=cfg.RUN_NAME + "_januspro_vlm",
        gradient_checkpointing=True,
        remove_unused_columns=False,
    )

    trainer_vlm = JanusRubricTrainer(
        model=vl_gpt,
        args=args,
        train_dataset=vlm['train'],
        eval_dataset=vlm.get('test', None),
        data_collator=identity_collator,
    )

    trainer_vlm.train()
    print('Training complete.')

In [ ]:
#@title VLM 5) Save Janus LoRA adapter (language_model) + processor
import os, json

vlm_adapter_dir = os.path.join(run_output_dir, 'adapter')
os.makedirs(vlm_adapter_dir, exist_ok=True)

# Save only the LoRA adapter weights (small)
vl_gpt.language_model.save_pretrained(vlm_adapter_dir)

# Save processor/tokenizer config so inference is reproducible
try:
    vl_chat_processor.save_pretrained(vlm_adapter_dir)
except Exception as e:
    print('VLChatProcessor save_pretrained not available; saving tokenizer only. Error:', e)
    tokenizer.save_pretrained(vlm_adapter_dir)

with open(os.path.join(vlm_adapter_dir, 'brainink_meta.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': cfg.BASE_MODEL,
        'mode': 'januspro_vlm',
    }, f, ensure_ascii=False, indent=2)

print('Saved VLM adapter/processor to:', vlm_adapter_dir)

In [ ]:
#@title VLM 6) Inference test on a single handwritten image (upload)
from google.colab import files
from peft import PeftModel
import torch

uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
img = Image.open(img_path).convert('RGB')

# Example rubric + prompt (replace with your real rubric)
rubric = "Total 10 points. (1) Correct slope (0-3). (2) Correct intercept (0-3). (3) Line drawn correctly (0-3). (4) Axes labeled (0-1)."
prompt = "Plot y = 2x + 1 for x in [-2,2]. Label intercept."

sys = "You are an expert teacher and grader. Grade strictly using ONLY the provided rubric. Return STRICT JSON only (no markdown)."
user = f"<image_placeholder>\nAssignment:\n{prompt}\n\nRubric:\n{rubric}\n\nStudent work is provided as image(s).\n\nTask: Produce a JSON grade with per-criterion scores, evidence, and feedback."

conversation = [
    {
        'role': 'User',
        'content': sys + "\n\n" + user,
        'images': [img],
    },
    {'role': 'Assistant', 'content': ''},
 ]

# Reload clean base + attach LoRA adapter to language_model
base = AutoModelForCausalLM.from_pretrained(
    cfg.BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=dtype,
 ).to(device).eval()

from janus.models import VLChatProcessor
processor = VLChatProcessor.from_pretrained(cfg.BASE_MODEL)
tok = processor.tokenizer

base.language_model = PeftModel.from_pretrained(base.language_model, vlm_adapter_dir)
base.language_model.eval()

prepare_inputs = processor(conversations=conversation, images=[img], force_batchify=True)
prepare_inputs = prepare_inputs.to(device, dtype=dtype) if hasattr(prepare_inputs, 'to') else prepare_inputs

inputs_embeds = base.prepare_inputs_embeds(**prepare_inputs)

with torch.no_grad():
    outputs = base.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tok.eos_token_id,
        bos_token_id=tok.bos_token_id,
        eos_token_id=tok.eos_token_id,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
    )

answer = tok.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
print(answer)